In [ ]:
!pip install -q chromadb openai pypdf

In [ ]:
import base64
import json

from IPython.display import Audio, Image

from chromadb import PersistentClient
from google.colab import userdata
from openai import OpenAI
from openai.types.responses import Response
from pathlib import Path
from pydantic import BaseModel, Field
from pypdf import PdfReader
from typing import List, Literal

In [ ]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

chroma_client = PersistentClient("/content/chromadb")
chroma_collection = chroma_client.get_or_create_collection("final_project")

In [ ]:
def extract_text(path_to_pdf: Path) -> str:
    with PdfReader(path_to_pdf) as pdf_reader:
        return "\n".join(pdf_page.extract_text() for pdf_page in pdf_reader.pages)

# TODO: Potentially extend with chunk by words/sentences.
def chunk_by_length(text: str, length: int = 1000, overlap: int = 300) -> List[str]:
    result = []

    index = 0
    while index < len(text):
        end = index + length
        result.append(text[index:end])
        index = end - overlap

    return result

In [ ]:
pdf_content = extract_text("/content/The Adventures of Sherlock Holmes.pdf")

chunks = chunk_by_length(pdf_content)
chunk_ids = list(str(i) for i in range(len(chunks)))

In [ ]:
# Advice: Fill the Chroma collection in batches
batch_index = 0
batch_length = 100
while batch_index < len(chunks):
    end = batch_index + batch_length
    chroma_collection.add(
        ids=chunk_ids[batch_index:end],
        documents=chunks[batch_index:end]
    )

    batch_index = end

In [ ]:
class QueryArgs(BaseModel):
    query: str = Field(description="A natural-language query.")

In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "query",
        "description": "Use this tool to get answers to questions against the underlying knowledgebase.",
        "parameters": {
            "type": "object",
            "properties": QueryArgs.model_json_schema()['properties'],
            "required": list(QueryArgs.model_fields.keys()),
            "additionalProperties": False
        },
        "strict": True
    }
]

In [ ]:
def query_handler(raw_args: str) -> str:
    query_args = QueryArgs.model_validate(json.loads(raw_args))
    query_result = chroma_collection.query(query_texts=[query_args.query], n_results=3)

    return "\n-----\n".join(doc for doc in query_result['documents'][0])


HANDLERS = {
    "query": query_handler
}

In [ ]:
def extract_tool_calls(response: Response):
    return [output_item for output_item in response.output if output_item.type == 'function_call']

def retrieve_information(prompt: str, verbose: bool = False) -> str:
    conversation = [
        { "role": "system", "content": "You are a helpful assistant that should ask questions about the underlying knowledgebase. Use the \"query\" tool aggresively with various arguments to achieve best results. Be very specific and do not suggest future work or questions." },
        { "role": "user", "content": prompt }
    ]

    ai_args = { "model": "gpt-5.4-mini", "reasoning": { "effort": "high" }, "tools": TOOLS }
    response = openai_client.responses.create(**ai_args, input=conversation)
    tool_calls = extract_tool_calls(response)

    while tool_calls:
        conversation.extend(response.output)

        if verbose:
            print(f"Found {len(tool_calls)} tools to execute.")

        for tc in tool_calls:
            handler = HANDLERS[tc.name]
            result = handler(tc.arguments)

            conversation.append({ "type": "function_call_output", "call_id": tc.call_id, "output": result })

        response = openai_client.responses.create(**ai_args, input=conversation)
        tool_calls = extract_tool_calls(response)

    return response.output_text

In [ ]:
class IntermediateResponse(BaseModel):
    question: str
    format: Literal['text', 'image', 'audio']

In [ ]:
def ask_ai(prompt: str, verbose: bool = False) -> None:
    intermediate_response = openai_client.responses.parse(
        model="gpt-5.4-mini",
        input=[
            { "role": "system", "content": "You are an intermediate AI agent that should determine the real question of a user against some book and the anticipated response format. The default response format is \"text\". If the user expects an \"image\" or \"audio\" to be generated, the question must predispose a creative and determined answer. The response format and the question are independent - the question should be irrelevant of details that expose what the response format is." },
            { "role": "user", "content": prompt }
        ],
        text_format=IntermediateResponse,
        reasoning={ "effort": "high" }
    )

    if verbose:
        print(intermediate_response.output_parsed)

    answer = retrieve_information(intermediate_response.output_parsed.question, verbose)

    if intermediate_response.output_parsed.format == 'text':
        print(answer)
    elif intermediate_response.output_parsed.format == 'image':
        image_content = openai_client.images.generate(
            prompt=f"This is the question I asked:\n{intermediate_response.output_parsed.question}\n\nThis is the answer I received:\n{answer}\n\nGenerate a realistic image inspired by the text.",
            model="gpt-image-2",
            output_format="png",
            size="1024x1024"
        )

        for image in image_content.data:
            image_bytes = base64.b64decode(image.b64_json)
            display(Image(data=image_bytes, format="png"))
    elif intermediate_response.output_parsed.format == 'audio':
        audio_content = openai_client.audio.speech.create(input=answer, model="tts-1-hd", voice="onyx")
        audio_content.write_to_file("/content/audio.mp3")
        display(Audio("/content/audio.mp3"))
    else:
        raise RuntimeError("Received incorrect response format.")

In [ ]:
ask_ai("Find quotes about dark or starry night.", verbose=True)

In [ ]:
ask_ai("Find at most two quotes about misteries. Then read them out loud to me - generate an audio.", verbose=True)

In [ ]:
ask_ai("Find at most two quotes about mysteries. Then generate an image to show them to me.", verbose=True)